In [0]:

# Recovery validation notebook parameters

dbutils.widgets.text("file_id", "")
dbutils.widgets.text("adf_run_id", "")
dbutils.widgets.text("batch_id", "")
dbutils.widgets.text("file_name", "")

file_id = dbutils.widgets.get("file_id").strip()
adf_run_id = dbutils.widgets.get("adf_run_id").strip()
batch_id = dbutils.widgets.get("batch_id").strip()
file_name = dbutils.widgets.get("file_name").strip()

if not all([file_id, adf_run_id, batch_id, file_name]):
    raise ValueError("All recovery parameters are required")

print("Recovery validation parameters initialized")


In [0]:

from pyspark.sql import functions as F
import json

control_rows = spark.sql("""
    SELECT source_file
    FROM dbw_healthcare_de_dev.etl.file_control
    WHERE file_id = :file_id
      AND adf_run_id = :adf_run_id
      AND batch_id = :batch_id
      AND file_name = :file_name
      AND pipeline_status = 'IN_PROGRESS'
      AND processing_stage = 'BRONZE'
      AND is_enabled = TRUE
""", args={
    "file_id": file_id,
    "adf_run_id": adf_run_id,
    "batch_id": batch_id,
    "file_name": file_name
}).collect()

if len(control_rows) != 1:
    raise RuntimeError(
        "Recovery rejected: file-control state or ownership mismatch"
    )

landing_path = control_rows[0]["source_file"]

bronze_path = landing_path.replace(
    "abfss://landing@", "abfss://bronze@", 1
)

if not landing_path.startswith("abfss://landing@"):
    raise RuntimeError("Unexpected source path in file_control")

if not any(
    f.path == bronze_path
    for f in dbutils.fs.ls(bronze_path.rsplit("/", 1)[0] + "/")
):
    raise RuntimeError("Recovery rejected: Bronze file does not exist")

existing_count = (
    spark.table("dbw_healthcare_de_dev.bronze.patients_raw")
    .filter(
        (F.col("file_id") == file_id)
        | (F.col("source_file") == bronze_path)
    )
    .limit(1)
    .count()
)

if existing_count:
    raise RuntimeError(
        "Recovery rejected: matching Bronze Delta records already exist"
    )

print("Recovery eligibility checks passed")

dbutils.notebook.exit(json.dumps({
    "recovery_eligible": True,
    "file_id": file_id,
    "batch_id": batch_id,
    "bronze_path": bronze_path
}))
